# Bolillero — Clase 11

Sortea quién pasa a recorrer su análisis de punta a punta. Correr las celdas en orden; para sortear otro alumno, volver a correr la última celda (los que ya salieron no se repiten).

Necesita `alumnos.xlsx` o `alumnos.csv` en esta carpeta (columnas "Nombre"/"Apellido", o el nombre completo en una columna).

In [ ]:
import random
import time
from pathlib import Path

import pandas as pd
from IPython.display import HTML, clear_output, display

# Si alguien faltó, anotalo acá (alcanza con parte del nombre):
AUSENTES = []

In [ ]:
# Cargar la lista
archivo = next((p for p in [Path('alumnos.xlsx'), Path('alumnos.csv')] if p.exists()), None)
if archivo is None:
    raise FileNotFoundError("Falta alumnos.xlsx o alumnos.csv en esta carpeta")

df_alumnos = pd.read_excel(archivo) if archivo.suffix == '.xlsx' else pd.read_csv(archivo)

cols = {c.lower().strip(): c for c in df_alumnos.columns}
col_nombre = next((cols[c] for c in cols if 'nombre' in c or 'name' in c), None)
col_apellido = next((cols[c] for c in cols if 'apellido' in c or 'surname' in c), None)

if col_nombre and col_apellido and col_nombre != col_apellido:
    nombres = (df_alumnos[col_nombre].astype(str).str.strip() + ' '
               + df_alumnos[col_apellido].astype(str).str.strip())
elif col_nombre:
    nombres = df_alumnos[col_nombre].astype(str).str.strip()
else:
    col_texto = next(c for c in df_alumnos.columns if df_alumnos[c].dtype == object)
    nombres = df_alumnos[col_texto].astype(str).str.strip()

todos = [n for n in nombres.dropna().unique().tolist() if n and n.lower() != 'nan']
presentes = [n for n in todos if not any(a.lower() in n.lower() for a in AUSENTES)]
ya_salieron = []

print(f"Lista cargada: {len(todos)} alumnos, {len(presentes)} presentes")

In [ ]:
# EL SORTEO — volver a correr esta celda para sacar otro alumno
candidatos = [n for n in presentes if n not in ya_salieron]
if not candidatos:
    raise RuntimeError("Ya salieron todos los presentes")

for paso in range(26):
    clear_output(wait=True)
    girando = random.choice(candidatos)
    display(HTML(f"<h1 style='text-align:center; color:#888; margin:60px 0;'>{girando}</h1>"))
    time.sleep(0.04 + paso * 0.014)  # arranca rápido, frena al final

elegido = random.choice(candidatos)
ya_salieron.append(elegido)

clear_output(wait=True)
display(HTML(f'''
<div style="text-align:center; border:3px solid #1a2e5a; border-radius:12px;
            padding:36px 20px; margin:10px 0; background:#f6f8fc;">
  <p style="font-size:15px; color:#666; margin:0;">Pasa a contar su análisis</p>
  <h1 style="font-size:44px; color:#1a2e5a; margin:12px 0;">{elegido}</h1>
  <p style="font-size:15px; color:#444; max-width:560px; margin:8px auto 0;">
    De punta a punta: qué encontraste en los datos, qué decidiste limpiar,
    con qué variables segmentaste, cuántos grupos elegiste y por qué,
    quiénes son tus segmentos... y a quién le mandás el cupón.
  </p>
</div>'''))